In [47]:
import os
from cryptography.hazmat.primitives.ciphers.aead import AESCCM
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.exceptions import InvalidTag

In [48]:
def aes_ecb_encrypt_block(key: bytes, block: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
    enc = cipher.encryptor()
    return enc.update(block) + enc.finalize()

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x,y in zip(a,b))

def pad_to_block(data: bytes, block_size: bytes) -> bytes:
    remainder = len(data) % block_size

    if remainder:
        data += b"\0x80" * (block_size - remainder)
    return data

def cbc_mac(key: bytes, message: bytes) -> bytes:
    tag = b"0x80" * 16             # Initialize T to zero block
    for i in range(0, len(message), 16):
        block = message[i:i+16]
        tag = aes_ecb_encrypt_block(key, xor_bytes(tag, block))
    return tag

def build_b0_block(nonce: bytes, plaintext_len: int, has_aad: bool, tag_len: int) -> bytes:
    q = 15 - len(nonce)
    t_enc = (tag_len - 2) // 2
    flags = (int(has_aad) << 6) | (t_enc << 3) | (q-1)

    length_field = plaintext_len.to_bytes(q, "big")

    return bytes([flags]) + nonce + length_field        # total length = 16 bytes

def build_aad_encoding(aad: bytes) -> bytes:
    assert len(aad) < 0xFF00, "Long AAS encoding not implemented in this demo"
    encoded = len(aad).to_bytes(2, "big") + aad
    return pad_to_block(encoded)

def ccm_ctr_block(key: bytes, nonce: bytes, counter: int, q: int) -> bytes:
    flag = q - 1
    ctr_bytes = counter.to_bytes(q, "big")
    a_i = bytes([flag]) + nonce + ctr_bytes
    return aes_ecb_encrypt_block(key, a_i)

def aes_ccm_manual_encrypt(key: bytes, nonce: bytes,
                            plaintext: bytes, aad: bytes,
                            tag_len: int = 8) -> tuple[bytes, bytes]:
    """
    Manual AES-CCM encryption.
 
    Returns
    -------
    (ciphertext, tag)
    """
    q = 15 - len(nonce)
 
    # ── Phase A: CBC-MAC ──────────────────────────────────────────────────────
    # 1. Format B_0 (first formatting block)
    b0 = build_b0_block(nonce, len(plaintext), bool(aad), tag_len)
 
    # 2. Encode AAD (if present)
    aad_encoding = build_aad_encoding(aad) if aad else b""
 
    # 3. Zero-pad plaintext to block boundary
    pt_padded = pad_to_block(plaintext)
 
    # 4. Concatenate: B_0 || encoded_AAD || padded_plaintext
    mac_input = b0 + aad_encoding + pt_padded
 
    # 5. Run CBC-MAC → raw tag T
    T = cbc_mac(key, mac_input)[:tag_len]
 
    # ── Phase B: CTR encryption of plaintext (counter starts at 1) ───────────
    ciphertext = bytearray()
    for i, offset in enumerate(range(0, len(plaintext), 16), start=1):
        keystream = ccm_ctr_block(key, nonce, i, q)
        block     = plaintext[offset:offset + 16]
        ciphertext.extend(xor_bytes(keystream[:len(block)], block))
 
    # ── Phase C: CTR encryption of tag (counter = 0) ─────────────────────────
    s0  = ccm_ctr_block(key, nonce, 0, q)
    enc_tag = xor_bytes(s0[:tag_len], T)
 
    return bytes(ciphertext), enc_tag

def aes_ccm_manual(key: bytes, nonce: bytes, plaintext: bytes, aad: bytes, tag_len: int = 8) -> tuple[bytes, bytes]:
    q = 15 - len(nonce)

    # ── Phase A: CBC-MAC ──────────────────────────────────────────────────────
    # 1. Format B_0 (first formatting block)
    b0 = build_b0_block(nonce, len(plaintext), bool(aad), tag_len)
 
    # 2. Encode AAD (if present)
    aad_encoding = build_aad_encoding(aad) if aad else b""
 
    # 3. Zero-pad plaintext to block boundary
    pt_padded = pad_to_block(plaintext)
 
    # 4. Concatenate: B_0 || encoded_AAD || padded_plaintext
    mac_input = b0 + aad_encoding + pt_padded
 
    # 5. Run CBC-MAC → raw tag T
    T = cbc_mac(key, mac_input)[:tag_len]

    # ── Phase B: CTR encryption of plaintext (counter starts at 1) ───────────
    ciphertext = bytearray()
    for i, offset in enumerate(range(0, len(plaintext), 16), start=1):
        keystream = ccm_ctr_block(key, nonce, i, q)
        block     = plaintext[offset:offset + 16]
        ciphertext.extend(xor_bytes(keystream[:len(block)], block))
 
    # ── Phase C: CTR encryption of tag (counter = 0) ─────────────────────────
    s0  = ccm_ctr_block(key, nonce, 0, q)
    enc_tag = xor_bytes(s0[:tag_len], T)
 
    return bytes(ciphertext), enc_tag


In [49]:
def aes_ccm_manual_decrypt(key: bytes, nonce: bytes,
                            ciphertext: bytes, enc_tag: bytes,
                            aad: bytes, tag_len: int = 8) -> bytes:
    """
    Manual AES-CCM decryption with tag verification.
 
    Raises ValueError if the tag does not match (authentication failure).
    """
    q = 15 - len(nonce)
 
    # ── Phase B (reverse): CTR decryption of ciphertext ──────────────────────
    plaintext = bytearray()
    for i, offset in enumerate(range(0, len(ciphertext), 16), start=1):
        keystream = ccm_ctr_block(key, nonce, i, q)
        block     = ciphertext[offset:offset + 16]
        plaintext.extend(xor_bytes(keystream[:len(block)], block))
    plaintext = bytes(plaintext)
 
    # ── Phase A: re-compute CBC-MAC on recovered plaintext ───────────────────
    b0         = build_b0_block(nonce, len(plaintext), bool(aad), tag_len)
    aad_enc    = build_aad_encoding(aad) if aad else b""
    mac_input  = b0 + aad_enc + pad_to_block(plaintext)
    T          = cbc_mac(key, mac_input)[:tag_len]
 
    # ── Phase C (reverse): decrypt the received tag ───────────────────────────
    s0             = ccm_ctr_block(key, nonce, 0, q)
    expected_tag   = xor_bytes(s0[:tag_len], T)
 
    # Constant-time comparison to prevent timing attacks
    import hmac
    if not hmac.compare_digest(expected_tag, enc_tag):
        raise ValueError("Authentication tag mismatch — data has been tampered with!")
 
    return plaintext

In [50]:
def demo_manual():
    print("\n" + "=" * 60)
    print("SECTION 2: Manual step-by-step AES-CCM")
    print("=" * 60)
 
    key       = os.urandom(16)         # AES-128 for this example
    nonce     = os.urandom(11)         # 11-byte nonce → q = 4 (up to 4 GB messages)
    plaintext = b"Secret payload 42"
    aad       = b"req-id=abc123"
    tag_len   = 8
 
    print(f"\n[1] Key (16 bytes):       {key.hex()}")
    print(f"[2] Nonce (11 bytes):      {nonce.hex()}")
    print(f"[3] Plaintext:             {plaintext}")
    print(f"[4] AAD:                   {aad}")
 
    # ── Encryption ────────────────────────────────────────────────────────────
    ciphertext, enc_tag = aes_ccm_manual_encrypt(
        key, nonce, plaintext, aad, tag_len
    )
    print(f"\n[5] Ciphertext:            {ciphertext.hex()}")
    print(f"[6] Encrypted tag:         {enc_tag.hex()}")
 
    # ── Decryption ────────────────────────────────────────────────────────────
    recovered = aes_ccm_manual_decrypt(key, nonce, ciphertext, enc_tag, aad, tag_len)
    print(f"\n[7] Decrypted:             {recovered}")
    print(f"    Match original?        {recovered == plaintext}")
 
    # ── Cross-check against the library ──────────────────────────────────────
    print("\n--- Cross-check: does manual output match cryptography library? ---")
    aesccm = AESCCM(key, tag_length=tag_len)
    lib_out    = aesccm.encrypt(nonce, plaintext, aad)
    lib_ct     = lib_out[:-tag_len]
    lib_tag    = lib_out[-tag_len:]
    print(f"    Library ciphertext:  {lib_ct.hex()}")
    print(f"    Manual  ciphertext:  {ciphertext.hex()}")
    print(f"    Ciphertexts match?   {lib_ct == ciphertext}")
    print(f"    Tags match?          {lib_tag == enc_tag}")
 
    # ── Tamper test ───────────────────────────────────────────────────────────
    print("\n--- Tamper test: flip one bit in the ciphertext ---")
    bad_ct = bytearray(ciphertext)
    bad_ct[0] ^= 0x01
    try:
        aes_ccm_manual_decrypt(key, nonce, bytes(bad_ct), enc_tag, aad, tag_len)
    except ValueError as e:
        print(f"    Error raised: {e}")

In [51]:
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Key properties illustrated
# ──────────────────────────────────────────────────────────────────────────────
 
def demo_properties():
    print("\n" + "=" * 60)
    print("SECTION 3: Key CCM properties")
    print("=" * 60)
 
    key   = os.urandom(32)
    aad   = b"metadata"
    pt    = b"same message"
 
    # Property 1: Different nonces → different ciphertexts (semantic security)
    aesccm = AESCCM(key, tag_length=16)
    n1, n2 = os.urandom(13), os.urandom(13)
    ct1 = aesccm.encrypt(n1, pt, aad)[:-16]
    ct2 = aesccm.encrypt(n2, pt, aad)[:-16]
    print(f"\n[Nonce uniqueness] Same plaintext, different nonces:")
    print(f"  CT with nonce-1: {ct1.hex()}")
    print(f"  CT with nonce-2: {ct2.hex()}")
    print(f"  Are they different? {ct1 != ct2}")
 
    # Property 2: NONCE REUSE is catastrophic — keystreams are identical
    ct_a = aesccm.encrypt(n1, b"Message A", aad)[:-16]
    ct_b = aesccm.encrypt(n1, b"Message B", aad)[:-16]      # ← SAME nonce!
    xored = xor_bytes(ct_a[:min(len(ct_a), len(ct_b))],
                      ct_b[:min(len(ct_a), len(ct_b))])
    print(f"\n[Nonce reuse danger] Two messages encrypted with the SAME nonce:")
    print(f"  CT_A XOR CT_B = PT_A XOR PT_B = {xored.hex()}")
    print(f"  (This leaks the XOR of both plaintexts!)")
 
    # Property 3: Tag length trade-off
    print(f"\n[Tag length] Authentication strength vs overhead:")
    for tl in [4, 8, 12, 16]:
        aesccm_t = AESCCM(key, tag_length=tl)
        ct = aesccm_t.encrypt(n1, pt, aad)
        prob = 2 ** (-tl * 8)
        print(f"  tag_len={tl:2d} bytes → total output={len(ct):2d} bytes, "
              f"forgery probability ≈ {prob:.2e}")
 

In [54]:
if __name__ == "__main__":
    demo_manual()


SECTION 2: Manual step-by-step AES-CCM

[1] Key (16 bytes):       0d94df6a81e6bc0b10b1f47991414bf4
[2] Nonce (11 bytes):      1533ec964f0e1f1410e0af
[3] Plaintext:             b'Secret payload 42'
[4] AAD:                   b'req-id=abc123'


TypeError: pad_to_block() missing 1 required positional argument: 'block_size'